The purpose of this code is to clean up our data and make it usable. It creates a pivot table so we can access the data easily and create new columns. Then it chooses the column we want, removes duplicates, and creates the columns we need so we can analyse such as met_performance and met_growth. 

In [1]:
import pandas as pd
import numpy as np
import os, csv, re
import matplotlib.pyplot as plt
from pathlib import Path
plt.close("all")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.0f}'.format) # Turn off scientific notation
pd.set_option('display.float_format', '{:.2f}'.format) # Force display floats with 2 decimal globally

def format_headers(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace(r"[./ >#:?*\-]", "_", regex=True)
        .str.replace(r"[()']", "", regex=True)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.lower()
        .str.replace(r"_+", "_", regex=True)
        .str.strip()
        .str.strip("_")
    )
    return df


from decimal import Decimal, ROUND_HALF_UP

In [2]:
# Selects which data set we want to clean

subject="math"
# subject="reading"
subject_title = subject.title()
subject_title

'Math'

In [3]:
# Opens the file that we want to use and read
df = pd.read_csv(f"exports/2526_diagnostic_results_{subject}_FAKEDATA.csv")
df = format_headers(df)
df.columns

Index(['student_id', 'student_last_name', 'student_first_name',
       'student_grade', 'school', 'completion_date', 'norming_window',
       'most_recent_diagnostic_ytd_y_n', 'overall_relative_placement',
       'percent_progress_to_annual_typical_growth'],
      dtype='str')

In [4]:
# Reduce dataframe to what we need

df = df[["student_id", "student_grade", "school", "norming_window", "most_recent_diagnostic_ytd_y_n",
          "overall_relative_placement", "percent_progress_to_annual_typical_growth"]]

In [5]:
# Checks the value counts of our data

for col in df.columns:
    
    
    print(df[col].value_counts(dropna = False))
    print()

student_id
1000006    4
1000021    4
1000038    4
1000040    4
1000045    4
          ..
1006024    1
1006050    1
1006056    1
1006060    1
1006076    1
Name: count, Length: 6081, dtype: int64

student_grade
10    1371
K     1360
1     1352
7     1314
5     1303
11    1303
12    1291
6     1288
8     1266
2     1263
9     1252
4     1229
3     1194
Name: count, dtype: int64

school
Central High               1176
Aspen Elementary           1107
South High                 1103
Heritage Middle            1086
Falcon Ridge Elementary    1078
Foothills Elementary       1074
North High                 1062
Horizon Middle             1057
Bear Creek Elementary      1044
Prairie Ridge Middle       1040
Mountain View Middle       1015
Eagle View Elementary      1013
East High                  1012
Cedar Hills Elementary     1011
Canyon Ridge Elementary     986
Grand Mesa Elementary       922
Name: count, dtype: int64

norming_window
Spring (March 2 - End of Year)            6176
Fall (Beginni

In [6]:
# Creates a copy of our data frame

aoy_df = df.copy()

In [7]:
# Check to see which grade levels exist

aoy_df["student_grade"].value_counts(dropna=False).sort_index()

student_grade
1     1352
10    1371
11    1303
12    1291
2     1263
3     1194
4     1229
5     1303
6     1288
7     1314
8     1266
9     1252
K     1360
Name: count, dtype: int64

In [8]:
# If we have extra grades showing it gets rid of those extras

aoy_df = aoy_df[aoy_df["student_grade"].isin(["K","1","2","3","4","5","6","7","8"])]

aoy_df["student_grade"].value_counts(dropna=False).sort_index()

student_grade
1    1352
2    1263
3    1194
4    1229
5    1303
6    1288
7    1314
8    1266
K    1360
Name: count, dtype: int64

In [9]:
# Creates met_performance column with base value 0 and assigns a 1 to all students that have met performance

aoy_df["met_performance"] = 0
aoy_df.loc[ (aoy_df["overall_relative_placement"] == "Mid or Above Grade Level") | 
    (aoy_df["overall_relative_placement"] == "Early On Grade Level"), "met_performance"] = 1

In [10]:
# sorts our columns so later we can get rid of duplicates

aoy_df = aoy_df.sort_values("most_recent_diagnostic_ytd_y_n", ascending = False)
aoy_df

,student_id,student_grade,school,norming_window,most_recent_diagnostic_ytd_y_n,overall_relative_placement,percent_progress_to_annual_typical_growth,met_performance
8369,1003031,4,Central High,Spring (March 2 - End of Year),Y,1 Grade Level Below,103,0
7547,1002739,4,Canyon Ridge Elementary,Spring (March 2 - End of Year),Y,2 Grade Levels Below,96,0
8615,1003122,8,Cedar Hills Elementary,Spring (March 2 - End of Year),Y,Early On Grade Level,232,1
8612,1003121,8,Foothills Elementary,Spring (March 2 - End of Year),Y,Early On Grade Level,93,1
8610,1003120,4,Grand Mesa Elementary,Spring (March 2 - End of Year),Y,2 Grade Levels Below,117,0
...,...,...,...,...,...,...,...,...
2903,1001055,8,South High,Winter (November 16 - March 1),N,Mid or Above Grade Level,79,1
8866,1003214,6,Bear Creek Elementary,Winter (November 16 - March 1),N,2 Grade Levels Below,0,0
8865,1003214,6,Bear Creek Elementary,Fall (Beginning of Year - November 15),N,1 Grade Level Below,0,0
8863,1003213,K,Eagle View Elementary,Winter (November 16 - March 1),N,3 or More Grade Levels Below,46,0


In [11]:
# Gets rid of duplicates based on student_id and norming_window

aoy_df = aoy_df.drop_duplicates(subset = ["student_id", "norming_window"], keep = "first")

In [12]:
# Fills in blanks with 0's in percent_progress_to_annual_typical_growth  

aoy_df = aoy_df.fillna({"percent_progress_to_annual_typical_growth" : 0})

In [13]:
# Since NA is gone changes the type to an integer

aoy_df["percent_progress_to_annual_typical_growth"] =  aoy_df["percent_progress_to_annual_typical_growth"].astype(int) 

In [14]:
# Creates a copy of just the end of year data

eoy_df = aoy_df[aoy_df["norming_window"] == "Spring (March 2 - End of Year)"].copy()

In [15]:
# Creates a copy of just the beginning of year data

boy_df = aoy_df[aoy_df["norming_window"] == "Fall (Beginning of Year - November 15)"].copy()

In [16]:
# Creates column in eoy_df, has_boy, and assigns a 1 if the student_id is in both eoy and boy and assigns a 0 if not

boy_students = set(boy_df["student_id"])

eoy_df["has_boy"] = np.where(eoy_df["student_id"].isin(boy_students), 1, 0)
eoy_df["has_boy"].value_counts(dropna=False)


has_boy
1    3464
0     527
Name: count, dtype: int64

In [17]:
# If student_id is not in eoy and boy sets to blank. Then assigns values of 1 and 0 if the students growth is >= 100

eoy_df['met_growth'] = np.where(
    eoy_df['has_boy'] == 0,
    np.nan,
    np.where(eoy_df['percent_progress_to_annual_typical_growth'] >= 100, 1, 0)
)
eoy_df['met_growth'].value_counts(dropna=False)

met_growth
1.00    1763
0.00    1701
NaN      527
Name: count, dtype: int64

In [18]:
# Creates a pivot table to sum and count the data

table = pd.pivot_table( eoy_df, values = ["met_performance", "met_growth"], index = ["school", "student_grade"], 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})

In [19]:
# Creates another pivot table with the same data but combined for the whole district

table2 = pd.pivot_table(eoy_df, values = ["met_performance", "met_growth"], index = "student_grade", 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})
table2["school"] = "Summit Valley School District"
table2 = table2.set_index("school",append=True)
table2 = table2.swaplevel(0,1)

In [20]:
# Calculates the met_growth percent for both tables

table["met_growth" , "percent"] = table["met_growth" , "sum"] / table["met_growth" , "count"]
table2["met_growth" , "percent"] = table2["met_growth" , "sum"] / table2["met_growth" , "count"]

In [21]:
# Calculates the met_performance percent for both tables

table["met_performance", "percent"] = table["met_performance", "sum"] / table["met_performance", "count"]
table2["met_performance", "percent"] = table2["met_performance", "sum"] / table2["met_performance", "count"]

In [22]:
# Puts the percent columns with the already existing met_growth and met_performance columns

table = table.sort_index(axis=1)
table2 = table2.sort_index(axis=1)

In [23]:
# Concatenates together table and table2

table = pd.concat([table, table2])
table.head()

met_growth               met_performance  \
                                    count percent   sum           count   
school           student_grade                                            
Aspen Elementary 1                     26    0.46 12.00              32   
                 2                     26    0.54 14.00              27   
                 3                     20    0.25  5.00              23   
                 4                     21    0.43  9.00              25   
                 5                     30    0.27  8.00              38   

                                            
                               percent sum  
school           student_grade              
Aspen Elementary 1                0.47  15  
                 2                0.67  18  
                 3                0.52  12  
                 4                0.52  13  
                 5                0.58  22

In [24]:
# Moves school index into df as a column

table = table.reset_index()
table.head()

school student_grade met_growth               met_performance  \
                                       count percent   sum           count   
0  Aspen Elementary             1         26    0.46 12.00              32   
1  Aspen Elementary             2         26    0.54 14.00              27   
2  Aspen Elementary             3         20    0.25  5.00              23   
3  Aspen Elementary             4         21    0.43  9.00              25   
4  Aspen Elementary             5         30    0.27  8.00              38   

               
  percent sum  
0    0.47  15  
1    0.67  18  
2    0.52  12  
3    0.52  13  
4    0.58  22

In [25]:
# Combines the stacked labels into 1. So met_growth and met_performance got divided into their percent, sum, and count

table.columns = [
    '_'.join(map(str, col)).strip()
    for col in table.columns
]
table.columns = table.columns.str.rstrip('_')

In [26]:
# Creates a folder

output_dir = Path("intermediate")
output_dir.mkdir(parents=True, exist_ok=True)

In [27]:
# Exports the table

table.to_csv(f"intermediate/01_clean_data_{subject}.csv", index=False)
print("Done")

Done
